# 00 · 데이터 준비 (한 번만 실행)

> 이 노트북은 **딱 한 번** 실행해 데이터를 **Google Drive에 받아둡니다.**
> 이후 모든 노트북·학습은 인터넷에서 다시 받지 않고 Drive에서 재사용합니다.

받는 데이터:
- **CholecSeg8k** (분할, ~3.1GB) → Drive에 캐시 (큰 파일 몇 개라 Drive에 무난)
- **Endoscapes2023** (CVS, zip ~수 GB) → **zip만 Drive에 보관**

⚠️ Endoscapes는 16만 개 파일이라 **Drive에 직접 풀면 몇 시간** 걸립니다. 그래서
*zip은 Drive에 저장*하고, 쓸 때마다 **로컬에 빠르게(수 분) 압축 해제**합니다.

## 0. 환경 준비

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh
print("준비 완료 ·", os.getcwd())

## 1. Google Drive 연결 (이 노트북은 필수)

`data/`·`outputs/`를 Drive에 연결하고 모델 캐시(`HF_HOME`)도 Drive로 둡니다.
처음 실행 시 인증 창이 뜹니다.

In [ ]:
from src.utils.colab import mount_drive
DRIVE = mount_drive()          # 예: /content/drive/MyDrive/surgical-cvs-ai
print("Drive project:", DRIVE)

## 2. CholecSeg8k → Drive

`data/`가 Drive에 연결돼 있어 다운로드가 Drive로 저장됩니다. 이미 받아뒀으면
캐시에서 즉시 끝납니다. (다음 세션부터는 mount_drive만 하면 자동 재사용 —
**재다운로드 없음**.)

In [ ]:
!bash scripts/download_cholecseg8k.sh

## 3. Endoscapes2023 → Drive (zip만 저장) + 로컬 압축 해제

CAMMA 공개 미러에서 zip을 **Drive에 한 번** 받습니다 (큰 파일 하나라 빠름).
그리고 학습용으로 **로컬에 압축 해제**합니다 — Drive에 16만 파일을 풀면 매우
느리지만, 로컬 풀기는 몇 분입니다. zip이 Drive에 있으니 *다시 다운로드할 일은
없습니다*.

In [ ]:
import os

ARCHIVE = f"{DRIVE}/endoscapes.zip"

# (1) zip을 Drive에 한 번만 다운로드 (이미 있으면 건너뜀)
if not os.path.exists(ARCHIVE):
    !wget -c -O "$ARCHIVE" https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
else:
    print("이미 Drive에 있음:", ARCHIVE)

# (2) 로컬로 빠르게 압축 해제 (매 세션 반복 — 로컬이라 빠름)
!mkdir -p /content/endoscapes2023
!unzip -q -n "$ARCHIVE" -d /content/endoscapes2023
print("Endoscapes 준비됨 → /content/endoscapes2023/endoscapes")

## 4. 둘 다 잘 불러와지나 확인

CholecSeg8k 3개 split + Endoscapes 3개 split 프레임 수가 0이 아니면 성공.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset
from src.data.endoscapes import Endoscapes2023Dataset

for s in ("train", "val", "test"):
    print(f"CholecSeg8k {s:5s}: {len(CholecSeg8kDataset(split=s))} frames")

for s in ("train", "val", "test"):
    ds = Endoscapes2023Dataset(root="/content/endoscapes2023", split=s)
    print(f"Endoscapes  {s:5s}: {len(ds)} CVS keyframes")

## 다음부터는 (요약)

- **CholecSeg8k**: Drive에 영구 저장됨 → 어떤 노트북이든 `mount_drive()` 후
  **자동 재사용** (재다운로드 없음). 분할 학습은 그대로:
  ```
  !python -m src.train.train_segmentation
  ```
- **Endoscapes**: zip이 Drive에 보관됨 → **매 세션 위 "3번 셀"로 로컬에 풀고**,
  CVS 학습 때 로컬 경로를 지정합니다:
  ```
  !python -m src.train.train_cvs data.root=/content/endoscapes2023
  ```
  (노트북 05에서 직접 쓸 땐 `Endoscapes2023Dataset(root="/content/endoscapes2023", ...)`)

이제 인터넷 재다운로드 없이 학습을 반복할 수 있습니다. 학습 체크포인트도
`outputs/`(Drive)에 저장돼 세션이 끊겨도 이어집니다.